# SWEAT tutorials: EVASPA processing

Explain how to estimate evapotranspiration with EVASPA model on a raster

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os
from pprint import pprint

In [ ]:
from sweat.api import filter_data_for_evaspa, read_input_data, run_evaspa
from sweat.evaspa.config import check_config_evaspa

In [ ]:
import sweat.extra.plot_notebook as pltn
from sweat.extra.utils_notebook import locate_test_data

In [ ]:
import logging

from sweat.logging import LoggerManager

LoggerManager.set_level(logging.INFO)

## Case: Simple execution

Input configuration

In [ ]:
config_dict = {
    "input": {
        "path": os.path.join(
            "data", "data_test_20230303T112730.tif"
        ),
        "date": "2023-03-03T11:27:30-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "ef": {
            "models": "default_evaspa",
        },
    },
}

#### Execution

Check input configuration

In [ ]:
config = check_config_evaspa(config_dict)

In [ ]:
pprint(config, sort_dicts=False)

Read

In [ ]:
input_data = read_input_data(config_dict["input"])

In [ ]:
pltn.plot_dataset(input_data)

Run 

In [ ]:
inst_xr, daily_xr, ws_xr = run_evaspa(input_data, config["params"])

In [ ]:
inst_xr

In [ ]:
daily_xr

In [ ]:
ws_xr

Visualize results

In [ ]:
pltn.plot_dataset(inst_xr)

In [ ]:
pltn.plot_flags(inst_xr, "flags")

In [ ]:
pltn.plot_dataset(daily_xr)

In [ ]:
pltn.plot_dataset(inst_xr)

## Case: Detailed configuration

#### Configuration

In [ ]:
config_dict = {
    "input": {
        "path": os.path.join(
            "data", "data_test_20230303T112730.tif"
        ),
        "date": "2023-03-03T11:27:30-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "filtering": {
            "cloud": {"op": "==", "value": 0},  # keep clear pixels
            "water": {"op": "==", "value": 0},  # keep non water pixels
            "qa": {"op": "==", "value": 1},  # keep pixels with qa = 1
            "ndvi": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep NDVI between [0,1]
            "fcover": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep Fcover between [0,1]
            "albedo": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep Fcover between [0,1]
            "height": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 300}]
            },  # Keep height between [0,300]
        },
        "ef": {
            "check": {"threshold": 0.02},
            "models": [
                {
                    "name": "model1",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": 2,
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": 2,
                            "selection": "median",
                        },
                    },
                    "var": "albedo",
                },
                {
                    "name": "model2",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": 5,
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "FlatEdge",
                    },
                    "var": "albedo",
                },
            ],
            "options": {
                "filtering": {
                    "albedo": {
                        "and": [
                            {"op": ">=", "value": 0.1},
                            {"op": "<=", "value": 0.3},
                        ]
                    }
                },
                "selection": False,
                "merging": {
                    "merging_method": "mean",
                    "uncertainty_method": "interquartile",
                },
            },
        },
        "seb": {
            "use_topo": True,
            "models": ["kustas"],
            "merging": {
                "merging_method": "mean",
                "uncertainty_method": "interquartile",
            },
        },
        "daily": {"use_topo": True, "method": "toa"},
        "waterstress": {"method": "ef"},
    },
}

#### Execution

Check input file

In [ ]:
config = check_config_evaspa(config_dict)

In [ ]:
pprint(config, sort_dicts=False)

Read

In [ ]:
input_data = read_input_data(config_dict["input"])

In [ ]:
pltn.plot_dataset(input_data)

Run

In [ ]:
inst_xr, daily_xr, inst_xr = run_evaspa(input_data, config_dict["params"])

Visualize results

In [ ]:
pltn.plot_dataset(inst_xr)

In [ ]:
pltn.plot_dataset(daily_xr)

In [ ]:
pltn.plot_dataset(ws_xr)

## Case: detailed pipeline

Additional imports

In [ ]:
from sweat.common import daily
from sweat.common import waterstress as ws
from sweat.evaspa import ef, seb
from sweat.evaspa.config import EVASPAParamsConfig

Configuration

In [ ]:
config_dict = {
    "input": {
        "path": os.path.join(
            "data", "data_test_20230303T112730.tif"
        ),
        "date": "2023-03-03T11:27:30-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "filtering": {
            "cloud": {"op": "==", "value": 0},  # keep clear pixels
            "water": {"op": "==", "value": 0},  # keep non water pixels
            "qa": {"op": "==", "value": 1},  # keep pixels with qa = 1
            "ndvi": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep NDVI between [0,1]
            "fcover": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep Fcover between [0,1]
            "albedo": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 1}]
            },  # Keep Fcover between [0,1]
            "height": {
                "and": [{"op": ">=", "value": 0}, {"op": "<=", "value": 300}]
            },  # Keep height between [0,300]
        },
        "ef": {
            "check": {"threshold": 0.02},
            "models": [
                {
                    "name": "model1",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "size",
                            "interval_size": 0.1,
                            "percentile": 2,
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "FlatEdge",
                    },
                    "var": "albedo",
                },
                {
                    "name": "model2",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "size",
                            "interval_size": 0.1,
                            "percentile": 2,
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "FlatRegressionEdge",
                        "config": {
                            "interval_type": "size",
                            "interval_size": 0.1,
                            "percentile": 1,
                            "selection": "median",
                        },
                    },
                    "var": "fcover",
                },
                {
                    "name": "model3",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "size",
                            "interval_size": 0.1,
                            "percentile": 2,
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "size",
                            "interval_size": 0.1,
                            "percentile": 1,
                            "selection": "median",
                        },
                    },
                    "var": "ndvi",
                },
            ],
            "options": {
                "filtering": {
                    "albedo": {
                        "and": [
                            {"op": ">=", "value": "percentile(1)"},
                            {"op": "<=", "value": "percentile(99)"},
                        ]
                    }
                },
                "selection": False,
                "merging": {
                    "merging_method": "mean",
                    "uncertainty_method": "interquartile",
                },
            },
        },
        "seb": {
            "use_topo": True,
            "models": ["kustas"],
            "merging": {
                "merging_method": "mean",
                "uncertainty_method": "interquartile",
            },
        },
        "daily": {"method": "geo", "name": "msg"},
        "waterstress": {"method": "ef"},
    },
}

Check configuration

In [ ]:
config = check_config_evaspa(config_dict)
# Validate parameters config
params_config = EVASPAParamsConfig.model_validate(config["params"])

In [ ]:
pprint(config, sort_dicts=False)

Read input data

In [ ]:
data = read_input_data(config["input"])

In [ ]:
data.data_vars

In [ ]:
data

In [ ]:
pltn.plot_dataset(data)

Initialize EF models

In [ ]:
models, options = ef.initialize(params_config.ef.model_dump(by_alias=True))

Filter data

In [ ]:
data = filter_data_for_evaspa(
    data=data,
    models=models,
    config=params_config.filtering.model_dump(by_alias=True),
)

In [ ]:
if (data["valid"].data == 0).sum() == 0:
    print("No invalid pixels")
else:
    pltn.plot_mask(data=data, mask="valid", invert=True)

In [ ]:
pltn.plot_flags(data, flags="flags")

Check variability

In [ ]:
ef.check_variability(
    data["lst"], mask=data["valid"], **params_config.ef.check.model_dump()
)

Compute EF

In [ ]:
ef_xr, inst_xr = ef.run(models, data, **options)

In [ ]:
ef_xr

In [ ]:
pltn.plot_models(data, ef_xr)

In [ ]:
pltn.plot_flags(ef_xr, flags="flags")

In [ ]:
pltn.plot_dataset(inst_xr)

Compute LE

In [ ]:
le_xr, merged_xr = seb.run(data, ef_xr, **params_config.seb.model_dump())
inst_xr = merged_xr.merge(
    inst_xr, join="override", compat="override", combine_attrs="no_conflicts"
)

In [ ]:
le_xr

In [ ]:
pltn.plot_dataset(le_xr)

In [ ]:
pltn.plot_flags(le_xr, flags="flags")

In [ ]:
inst_xr

In [ ]:
pltn.plot_dataset(inst_xr)

Extrapolate at daily scale

In [ ]:
daily_xr = daily.extrapolate_at_daily_scale(
    inst_xr,
    variables=["le", "et", "uncertainty_le", "uncertainty_et"],
    extra=data,
    **params_config.daily.model_dump(),
)

In [ ]:
pltn.plot_dataset(daily_xr)

Compute water stress indices

In [ ]:
ws_xr = ws.run(et=inst_xr, **params_config.waterstress.model_dump())

In [ ]:
ws_xr

In [ ]:
pltn.plot_dataset(ws_xr)